# Part 1: Data Acquisition and Preprocessing Pipeline

In this notebook, we will walk through the **first stage of building a GitLab Rag Chatbot**.  
This stage focuses on transforming raw data from GitLab into a structured, searchable knowledge base.

The key steps we will cover include:

- **Data Acquisition**  
  Collecting raw content from GitLab’s *Direction* webpage and *Handbook repository*.  

- **Preprocessing**  
  Cleaning the extracted HTML/text (removing scripts, styles, headers, footers, etc.) and saving it in organized `.txt` files.  

- **Chunking**  
  Splitting large text into semantically meaningful pieces  (`750-character chunks with 150 overlap`) to preserve context.  

- **Embedding**  
  Converting text chunks into dense vector representations using HuggingFace `SentenceTransformers (all-MiniLM-L6-v2)`.  

- **Vector Storage**  
  Indexing embeddings with **FAISS** for efficient similarity search and retrieval in later chatbot stages.  

---
##Expected Outcomes
📌
 By the end of this notebook, we will have:  
- A clean, structured dataset (`handbook_cleaned_FULL.txt`, `direction_final.txt`)  
- Chunks of text prepared for semantic search
- Embeddings generated and stored  
- A FAISS index ready for chatbot integration  

This pipeline forms the **foundation** of the chatbot, enabling advanced steps like RAG (Retrieval-Augmented Generation) in subsequent parts.




## Imports and Setup

In [ ]:
# Install required libraries
!pip install requests beautifulsoup4
!pip install langchain langchain-community langchain-huggingface faiss-cpu


In [ ]:
import requests # `requests` → to fetch web content (HTML)
from bs4 import BeautifulSoup # BeautifulSoup → to parse and clean unwanted tags
import os # os → for file handling


# Stage 1: Scraping Raw Content From Gitlab

###Define Target URL & Output Paths

We save the cleaned text into "**data/Direction/direction_cleaned.txt.**"
This ensures reproducibility and maintains a proper project folder structure.

In [ ]:
# URL to scrape
url = "https://about.gitlab.com/direction/"

# Directory and output file setup
output_dir = "data/Direction"
output_file = os.path.join(output_dir, "direction_cleaned.txt")

os.makedirs(output_dir, exist_ok=True)

###Cleaning Function

This function ensures:
- Unnecessary elements (scripts, navigation menus, headers/footers) are removed
- Text remains **clean, structured, and human-readable**

In [ ]:
def clean_html(html):
    """
    Remove unnecessary HTML elements like <script>, <style>, <header>, <footer>, and <nav>.
    Extracts meaningful textual content only.
    """
    soup = BeautifulSoup(html, "html.parser")

    # Remove unwanted tags
    for tag in soup(["script", "style", "footer", "header", "nav"]):
        tag.decompose()

    # Extract visible text
    return soup.get_text(separator="\n", strip=True)


###Scraping GitLab Direction Page

We send an HTTP request to GitLab's Direction page.  
`raise_for_status()` guarantees that errors (e.g., 404 or 500) are immediately caught.

In [ ]:
response = requests.get(url)
response.raise_for_status()   # Raises error if request fails

cleaned_text = clean_html(response.text)

###Saving and Preview Data

Saving data to `data/Direction/direction_cleaned.txt`

In [ ]:
with open(output_file, "w", encoding="utf-8") as f:
    f.write("## SECTION: GitLab Direction Main Page\n\n")
    f.write(cleaned_text)

print(f"✅ Saved cleaned Direction content to: {output_file}")

# Preview the first 15 lines
with open(output_file, "r", encoding="utf-8") as f:
    for i in range(15):
        print(f.readline().strip())



# Stage 2: Enhancement Of Direction Data

In this stage, we improve upon the **initial cleaned file** produced ear;y (`direction_cleaned.txt`).  

Why?  
- The scraped version may still miss certain parts of the page or include unwanted noise.  
- To fix this, we **merge** two sources:
  - `direction_cleaned.txt` → cleaned scrape output  
  - `raw_direction.txt` → raw dump (to catch any missed content)  

Steps performed:  
1. Load both files.  
2. Normalize paragraphs (remove extra whitespace, bullets, and GitLab boilerplate).  
3. Remove duplicates and filter out very short/noisy text.  
4. Merge into a **final enhanced dataset** → `direction_final.txt`.

### Step 1: Setup & Input Paths

We define where our input files are located:
- `direction_cleaned.txt` → output from Stage 1  
- `raw_direction.txt` → raw dump of the Direction page  

We also set up the output path for our final enhanced file:  
- `direction_final.txt`


⚠️ **Note:**  
If running on **Google Colab/Jupyter online**, upload `content.rar`(Includes Raw downloaded handbook and direction data for comparison and enhancement) and run this cell to extract the Handbook.  
Skip if you already have `data/Handbook/content/` locally.


In [ ]:
# Install unrar (if not already available)
!apt-get install -y unrar

# Make the Handbook folder
!mkdir -p data/Handbook

# Extract the RAR file into data/Handbook/
!unrar x content.rar data/Handbook/

In [ ]:
from pathlib import Path
import re

# Input files
file1 = Path("data/Direction/direction_cleaned.txt")   # Stage 1 cleaned file
file2 = Path("/content/data/Handbook/content/Raw_data/raw_direction.txt")  # Raw dump

# Output file
final_output = Path("data/Direction/direction_final.txt")
final_output.parent.mkdir(parents=True, exist_ok=True)

### Step 2: Define Cleaning Functions

We create two helper functions:
1. `clean_paragraph()` → removes extra spaces, unwanted boilerplate text, and bullet markers.
2. `get_cleaned_paragraphs()` → splits the text into paragraphs, cleans them, and filters out very short ones (< 50 characters).


In [ ]:
def clean_paragraph(para: str) -> str:
    """
    Cleans a paragraph by:
    - Normalizing whitespace
    - Removing GitLab boilerplate text
    - Stripping bullet characters
    """
    para = para.strip()
    para = re.sub(r"\s+", " ", para)  # Normalize whitespace
    para = re.sub(r"©.*GitLab.*|Edit this page|Contact us|Get free trial", "", para, flags=re.IGNORECASE)
    para = para.strip("•")  # Remove bullet characters
    return para.strip()

def get_cleaned_paragraphs(text: str) -> set:
    """
    Splits text into paragraphs, cleans each,
    and filters out very short or noisy content.
    """
    paras = text.split("\n\n")
    return set(clean_paragraph(p) for p in paras if len(clean_paragraph(p)) > 50)


### Step 3: Load and Clean Both Sources

We now read both input files: Cleaned and Dump file

Each file is processed into a **set of cleaned paragraphs**, ensuring no duplicates within a single file.


In [ ]:
# Read both versions of the text
text1 = file1.read_text(encoding="utf-8")
text2 = file2.read_text(encoding="utf-8")

# Clean into sets of paragraphs
paras1 = get_cleaned_paragraphs(text1)
paras2 = get_cleaned_paragraphs(text2)

print(f"✅ direction_cleaned.txt paragraphs: {len(paras1)}")
print(f"✅ raw_direction.txt paragraphs: {len(paras2)}")


### Step 4: Merge and Deduplicate

We combine the two sets of paragraphs (from cleaned and raw data).  
Using a **set union** ensures that duplicates are removed automatically.  
Finally, we sort the paragraphs to maintain a consistent order.


In [ ]:
# Combine both sources and sort
merged_paras = sorted(paras1.union(paras2))

print(f"📄 Total unique, cleaned paragraphs: {len(merged_paras)}")


### Step 5: Save Final Enhanced File

The fully enhanced and deduplicated paragraphs are now saved into:
- `direction_final.txt`  

This file will serve as the **final cleaned dataset** for the GitLab Direction page.


In [ ]:
with open(final_output, "w", encoding="utf-8") as f:
    f.write("## SECTION: GitLab Direction (Enhanced)\n\n")
    f.write("\n\n".join(merged_paras))

print(f"✅ Final merged Direction file saved to: {final_output}")


### Step 6: Preview Final Output

We show the first 20 lines of `direction_final.txt` to validate:
- No leftover boilerplate text
- Paragraphs are properly formatted
- Data looks ready for downstream tasks


In [ ]:
# Show the first 20 lines of the final file
with open(final_output, "r", encoding="utf-8") as f:
    for i in range(20):
        print(f.readline().strip())


# Stage 3: Extracting & Structuring the GitLab Handbook

After preparing the GitLab Direction data, the next step is to extract the **GitLab Handbook**.  

The Handbook is hosted as a collection of Markdown files. Unlike a single webpage, it is **hierarchical** (with folders and subfolders).  
To prepare this for our chatbot, we must:  

1. Walk through all Markdown files in the Handbook directory  
2. Clean each file (remove frontmatter, links, formatting)  
3. Preserve the folder structure as **section headers**  
4. Save the final dataset in a structured format → `handbook_structured.txt`

### Step 1: Setup & File Paths

We define where the GitLab Handbook files are located (`raw_data/Handbook/content`) and where the cleaned output should be saved (`data/Handbook/handbook_Cleaned.txt`).



In [ ]:
import os
import re
from pathlib import Path

HANDBOOK_DIR = "data/Handbook/content"
# Where the cleaned + structured file will be saved
OUTPUT_FILE = "data/Handbook/handbook_cleaned.txt"


### Step 2: Cleaning Markdown Files

We define a function to clean Markdown text by:
- Removing YAML frontmatter (`--- ... ---`)  
- Removing images  
- Converting links into plain text  
- Stripping inline code, bold, italic, and headers  

This ensures we keep **only the meaningful content** from each file.


In [ ]:
def clean_markdown(text: str) -> str:
    text = re.sub(r"^---\s*\n.*?\n---\s*", "", text, flags=re.DOTALL | re.MULTILINE)  # YAML frontmatter
    text = re.sub(r"!\[.*?\]\(.*?\)", "", text)  # Remove images
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)  # Keep link text only
    text = re.sub(r"`{1,3}(.*?)`{1,3}", r"\1", text)  # Inline code
    text = re.sub(r"[*_]{1,3}", "", text)  # Bold/italic markers
    text = re.sub(r"^#+\s*", "", text, flags=re.MULTILINE)  # Strip headings
    return text.strip()


### Step 3: Logical Section Paths

We compute a **logical section path** for each file, based on its folder.  
Example:  
`handbook/engineering/ci.md` → `engineering/ci.md`  

This makes it clear later which section each piece of content came from.


In [ ]:
def get_section_path(root, file):
    relative_path = os.path.relpath(os.path.join(root, file), HANDBOOK_DIR)
    return relative_path.replace("\\", "/")


### Step 4: Extract All Handbook Files

We walk through all Markdown files in the Handbook, clean their text, and wrap each one in a structured block:



In [ ]:
def extract_handbook_structured():
    collected_sections = []
    count = 0

    for root, _, files in os.walk(HANDBOOK_DIR):
        files = sorted(f for f in files if f.endswith(".md"))
        for file in files:
            file_path = os.path.join(root, file)
            try:
                raw = Path(file_path).read_text(encoding="utf-8")
                cleaned = clean_markdown(raw)
                section_path = get_section_path(root, file)

                block = f"\n\n---\n# SECTION: {section_path}\n\n{cleaned}"
                collected_sections.append(block)
                count += 1
            except Exception as e:
                print(f"❌ Could not process {file_path}: {e}")

    print(f"✅ Processed {count} Markdown files into structured format.")
    return "\n".join(collected_sections)


### Step 5: Save Final Structured Handbook

We combine all sections into a single file:
- `handbook_structured.txt`  

This dataset preserves the Handbook’s structure and content, making it ready for chunking and embeddings.


In [ ]:
Path(os.path.dirname(OUTPUT_FILE)).mkdir(parents=True, exist_ok=True)
structured_text = extract_handbook_structured()

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(structured_text)

print(f"✅ Saved structured Handbook content to {OUTPUT_FILE}")


# Stage 4: Comparing Handbook Versions

After extracting and cleaning the GitLab Handbook, we sometimes generate multiple versions (e.g., one from raw extraction, another after merging or enhancing).  

This script compares two versions to check:

- **Basic statistics** (line and word counts)  
- **Section headers** (how many `## SECTION:` markers exist, and which are unique to each version)  
- **Paragraph-level differences** (common vs unique paragraphs)  
- **Overall similarity score** (using `SequenceMatcher`)  

This helps validate whether enhancements added meaningful content or just duplicates.

### Step 1: Imports & File Paths

We import the required libraries (`Path`, `SequenceMatcher`) and define the paths of the two versions we want to compare:
- `handbook_Cleaned.txt` → structured version we extracted  
- `handbook_cleaned_FULL.txt` → enhanced or merged version  


In [ ]:
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Define paths to the two versions
file1 = Path("/content/data/Handbook/content/Raw_data/handbook_cleaned_FULL.txt")
file2 = Path("/content/data/Handbook/handbook_cleaned.txt")



### Step 2: Load Both Files

We define a helper function `load_text()` to read the contents of each file into memory.


In [ ]:
def load_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

# Load both versions
text1 = load_text(file1)
text2 = load_text(file2)


### Step 3: Basic Statistics

We calculate:
- Number of lines  
- Number of words  

This gives us a quick overview of file size differences.


In [ ]:
lines1 = text1.splitlines()
lines2 = text2.splitlines()

words1 = text1.split()
words2 = text2.split()

print("🔍 Basic Stats:")
print(f"{file1.name}: {len(lines1)} lines | {len(words1)} words")
print(f"{file2.name}: {len(lines2)} lines | {len(words2)} words")


### Step 4: Section Header Comparison

Each handbook file uses `## SECTION:` as markers.  
We compare:
- Total number of sections in each version  
- Sections unique to the FULL version (added later)


In [ ]:
headers1 = set(line for line in lines1 if line.startswith("## SECTION:"))
headers2 = set(line for line in lines2 if line.startswith("## SECTION:"))

print("\n📁 Section Header Comparison:")
print(f"{file1.name}: {len(headers1)} sections")
print(f"{file2.name}: {len(headers2)} sections")
print(f"New sections added in FULL: {len(headers2 - headers1)}")


### Step 5: Paragraph-Level Comparison

We split text into paragraphs (separated by blank lines) and compare:
- Common paragraphs  
- Unique to file1 (extracted version)  
- Unique to file2 (FULL version)


In [ ]:
paras1 = set(text1.split("\n\n"))
paras2 = set(text2.split("\n\n"))

common = paras1 & paras2
only_in_file1 = paras1 - paras2
only_in_file2 = paras2 - paras1

print("\n📄 Paragraph-Level Comparison:")
print(f"Common Paragraphs: {len(common)}")
print(f"Unique in {file1.name}: {len(only_in_file1)}")
print(f"Unique in {file2.name}: {len(only_in_file2)}")


### Step 6: Overall Similarity Score  

We use **TF-IDF with Cosine Similarity** to compare the two handbook versions.  
This method captures word importance and gives a more reliable similarity score than simple text matching.  



In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)  # limit features for speed
tfidf = vectorizer.fit_transform([text1, text2])

cos_sim = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
print(f"📊 Cosine Similarity (TF-IDF): {cos_sim:.4f}")

# Stage 5: Chunking, Embeddings & Vector Store  

In this stage, we prepare our data for retrieval-augmented generation (RAG).  
The process involves:  
1. **Splitting documents into manageable chunks** → ensures that each chunk is small enough for semantic search.  
2. **Generating embeddings** → converting text chunks into dense numerical vectors using `all-MiniLM-L6-v2`.  
3. **Storing vectors in FAISS** → enabling fast similarity search for chatbot queries.  

By the end, we will have a **FAISS index**


In [ ]:
# Install required packages
!pip install langchain langchain-community langchain-huggingface faiss-cpu


In [ ]:
#Imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from pathlib import Path
# Build FAISS vector database
from tqdm import tqdm


### Step 1: Load Cleaned Data  

We load the cleaned Handbook and Direction files produced from earlier stages.  

In [ ]:
# Load cleaned content
handbook_text = Path("/content/data/Handbook/handbook_cleaned.txt").read_text(encoding="utf-8")
direction_text = Path("/content/data/Direction/direction_final.txt").read_text(encoding="utf-8")


### Step 2: Initialize Text Splitter  

We use `RecursiveCharacterTextSplitter` to break text into chunks of ~750 characters  
with an overlap of 150 to preserve context across splits.  


In [ ]:
# Initialize text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=150,
    length_function=len,
)


### Step 3: Chunk Text with Metadata  

Each chunk is tagged with its **source** (Handbook or Direction) and its **section header**.  
This metadata helps the chatbot provide more contextual answers.  


In [ ]:
def chunk_with_metadata(text: str, source_label: str):
    """Split text into chunks with metadata."""
    sections = text.split("## SECTION:")
    documents = []

    for section in sections:
        if not section.strip():
            continue
        header, *content = section.strip().split("\n", 1)
        body = content[0] if content else ""
        chunks = splitter.create_documents([body])
        for chunk in chunks:
            chunk.metadata = {
                "source": source_label,
                "section": header.strip()
            }
        documents.extend(chunks)
    return documents

# Chunk both sources
handbook_docs = chunk_with_metadata(handbook_text, "handbook")
direction_docs = chunk_with_metadata(direction_text, "direction")
all_docs = handbook_docs + direction_docs
print(f"✅ Total chunks created: {len(all_docs)}")


### Step 4: Generate Embeddings  

We use **HuggingFace MiniLM (all-MiniLM-L6-v2)** to embed each chunk into a vector space.  
These embeddings capture semantic meaning, enabling similarity-based retrieval.  


In [ ]:
# Embedding model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

### Step 5: Build & Save FAISS Index  

We embed the text in **batches** and build a FAISS vector store, saving it.  
This index will later enable fast semantic search for the chatbot.  


In [ ]:
# Build FAISS vector database with progress
vectordb = None
batch_size = 200  # adjust depending on Colab speed

for i in tqdm(range(0, len(all_docs), batch_size), desc="🔄 Embedding chunks"):
    batch = all_docs[i:i+batch_size]
    if vectordb is None:
        vectordb = FAISS.from_documents(batch, embedding_model)
    else:
        vectordb.add_documents(batch)

# Save locally
vectordb.save_local("/content/data")
print("✅ FAISS index saved to: /content/data")
